# `c03_e12` — Twelve-Month Enrollment and Instructional Activity

**Component curation notebook.** Fetches the raw IPEDS distribution files, verifies the
reference period against official documentation, locks the schema, reshapes to the
declared grain, validates, and writes one curated table with a metadata sidecar.

| Property | Value |
|---|---|
| Native tables | `EFFY2023`, `EFIA2023`, `DRVEF122023` |
| Reference period | July 1, 2022 through June 30, 2023 |
| Curated grain | `UNITID` x `EFFYALEV` |
| Output | `data/curated/c03_e12.parquet` |

Twelve-month unduplicated headcount is the right denominator for cost-per-student measures because it matches the fiscal year that Finance reports on. Fall census counts do not, and mixing the two understates cost per student at institutions with heavy summer enrollment.

The grain is `UNITID` x `EFFYALEV`, which carries 27 distinct level codes. `EFFYLEV` (4 codes) and `LSTUDY` (3 codes) are coarser rollups of the same records, so the file contains totals alongside their own components. Select the level rows you need; never sum across them.

> **Pitfall.** Do not assume EFYTOTLM + EFYTOTLW == EFYTOTLT. Gender-inclusive reporting categories mean the men/women columns no longer partition the total. Any men/women share must use EFYGUKN as its denominator, or shares will quietly sum to under 100 percent at exactly the institutions using the newer categories.

## 1. Environment

One import surface, so a parsing quirk is fixed once rather than twelve times.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import ipeds_utils as iu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

SLUG = "c03_e12"
TABLES = ['EFFY2023', 'EFIA2023', 'DRVEF122023']
GRAIN = ['UNITID', 'EFFYALEV']
REFERENCE_PERIOD = 'July 1, 2022 through June 30, 2023'

print("ipeds_utils", iu.__version__, "| pandas", pd.__version__)

ipeds_utils 1.1.0 | pandas 3.0.5


## 2. Retrieve

Downloads are cached, so re-running this notebook is offline and cheap. Every retrieval returns a provenance record carrying a SHA-256 digest, which is what makes a result reproducible rather than merely repeatable.

In [2]:
RAW_DIR = "../data/raw"   # relative to notebooks/, so all twelve share one cache

provenance = [iu.fetch(t, raw_dir=RAW_DIR) for t in TABLES]
pd.DataFrame(provenance)[["table", "data_bytes", "data_sha256", "retrieved_utc"]]

,table,data_bytes,data_sha256,retrieved_utc
0,EFFY2023,7295855,bfc36572b930551f784ebb9c1e428b8f985fa70a80dab2...,2026-09-24T17:19:13+00:00
1,EFIA2023,191187,87229fac9a64563df477b47f392d5af7913e474363b440...,2026-09-24T17:19:13+00:00
2,DRVEF122023,659607,183a6b886a4558d0334c2048a3acf1ffd8189fcd8d1ed9...,2026-09-24T17:19:13+00:00


## 3. Verify the reference period

**Do not skip this cell.** The filename year is not the reference period, and the offsets are not uniform across components. This assertion fails loudly rather than letting a misaligned period corrupt every downstream year comparison, where it would be invisible in the data itself.

In [3]:
intro = iu.assert_reference_period(
    provenance[0]["dict_path"],
    expect=r'(July 1, 2022|2022-23)',
    table=TABLES[0],
)
print(intro[:600])

File Documentation for the 12-month Unduplicated Head Count Data File, 2022-23
(Final/revised release)
Filename EFFY2023
Provisional release: August 2024
Filename EFFY2023_RV
Final/revised release: September 2025
Filename EFFY2023
Overview This file contains the unduplicated headcount of students enrolled over the 12-month period (July 1, 2022 - June 30, 2023) for both undergraduate and graduate levels. Beginning with the 2019-20 data collected in 2020-21, undergraduate level headcounts were available by attendance status (full- and part-time) for both degree/certificate-seeking and nondegree/


## 4. Inspect the dictionary

Variable labels come from the published dictionary, never from memory. This is also where value sets are read, so categorical decoding is driven by the official codebook and a taxonomy revision surfaces as unmatched codes instead of a plausible-looking wrong label.

In [4]:
variables = iu.read_dict(provenance[0]["dict_path"])
valuesets = iu.read_valuesets(provenance[0]["dict_path"])

print(f"{len(variables)} variables documented, {len(valuesets)} value-set rows")
variables[["varname", "vartitle"]].head(20)

38 variables documented, 34 value-set rows


,varname,vartitle
0,UNITID,Unique identification number of the institution
1,EFFYALEV,Level and degree/certificate-seeking status of...
2,EFFYLEV,Undergraduate or graduate level of student
3,LSTUDY,Original level of study on survey form
4,EFYTOTLT,Grand total
5,EFYTOTLM,Grand total men
6,EFYTOTLW,Grand total women
7,EFYAIANT,American Indian or Alaska Native total
8,EFYAIANM,American Indian or Alaska Native men
9,EFYAIANW,American Indian or Alaska Native women


## 5. Load and lock the schema

The first run records the column signature; later runs fail if it drifts.

In [5]:
KEEP = ['UNITID', 'EFFYALEV', 'EFFYLEV', 'LSTUDY', 'EFYTOTLT', 'EFYTOTLM', 'EFYTOTLW', 'EFYGUKN', 'EFYGUAN', 'EFYAIANT', 'EFYASIAT', 'EFYBKAAT', 'EFYHISPT', 'EFYNHPIT', 'EFYWHITT', 'EFY2MORT', 'EFYUNKNT', 'EFYNRALT']

raw = iu.read_csv(provenance[0]["data_path"])
print("raw shape", raw.shape)

lock = iu.lock_schema(raw, TABLES[0], schema_dir="../schemas", strict=False)
print("schema:", lock["status"], "| added", lock["added"][:5], "| removed", lock["removed"][:5])

available = [c for c in KEEP if c in raw.columns]
missing = [c for c in KEEP if c not in raw.columns]
if missing:
    print("NOT PRESENT in this cycle (verify against the varlist above):", missing)

frame = raw[available].copy()
frame.head()

raw shape (116437, 72)
schema: unchanged | added [] | removed []


,UNITID,EFFYALEV,EFFYLEV,LSTUDY,EFYTOTLT,EFYTOTLM,EFYTOTLW,EFYGUKN,EFYGUAN,EFYAIANT,EFYASIAT,EFYBKAAT,EFYHISPT,EFYNHPIT,EFYWHITT,EFY2MORT,EFYUNKNT,EFYNRALT
0,100654,1,1,999,6627,2644,3983,6622.0,NaN,16,18,5633,72,10,142,74,541,121
1,100654,2,2,1,5664,2329,3335,5662.0,NaN,13,13,5070,67,10,119,73,234,65
2,100654,3,-2,1,5660,2329,3331,NaN,NaN,13,13,5067,67,10,118,73,234,65
3,100654,4,-2,1,1756,722,1034,NaN,NaN,3,2,1573,25,4,23,24,79,23
4,100654,5,-2,1,3904,1607,2297,NaN,NaN,10,11,3494,42,6,95,49,155,42


## 6. Mask reserved missing codes

IPEDS encodes missingness as negative integers. A mean computed without masking them is badly wrong and looks entirely plausible, which is what makes this the most costly single omission in IPEDS analysis.

In [6]:
RESERVED = [-1, -2, -3, -9]

numeric_cols = [
    c for c in frame.columns
    if c not in ("UNITID", *GRAIN) and pd.api.types.is_numeric_dtype(frame[c])
]

before = frame[numeric_cols].isna().sum().sum()
for col in numeric_cols:
    frame.loc[frame[col].isin(RESERVED), col] = np.nan
after = frame[numeric_cols].isna().sum().sum()

# Masking turns an integer column into float (1 becomes 1.0). Measures can stay float,
# since NaN is what the models expect, but category codes go back to nullable integers
# so they print, join, and decode as codes rather than as 1.0.
for col in ['EFFYLEV']:
    if col in frame.columns and pd.api.types.is_float_dtype(frame[col]):
        if (frame[col].dropna() % 1 == 0).all():
            frame[col] = frame[col].astype("Int64")

print(f"masked {after - before:,} reserved-code cells across {len(numeric_cols)} numeric columns")

masked 102,738 reserved-code cells across 16 numeric columns


## 7. Carry the imputation flags

An imputed value and a reported value are not the same evidence. A column where most institutions carry a generated flag should not be modelled as though it were observed, and this is where that judgement becomes possible.

In [7]:
values, flags = iu.split_imputation_flags(raw, numeric_cols)

if flags.shape[1] > 1:
    summary = iu.imputation_summary(flags)
    display(summary.head(15))
    reported = summary[summary.flag == "R"].set_index("column")["share"]
    weak = reported[reported < 0.90]
    if len(weak):
        print("Columns under 90% reported — interpret with care:")
        display(weak)
else:
    print("No X-prefixed imputation flags accompany this file.")

,column,flag,n,share
3,XEFYGUAN,A,112546,0.9666
4,XEFYGUAN,R,2490,0.0214
5,XEFYGUAN,S,1399,0.0120
6,XEFYGUAN,C,2,0.0000
0,XEFYGUKN,A,102738,0.8823
1,XEFYGUKN,R,12300,0.1056
2,XEFYGUKN,S,1399,0.0120


Columns under 90% reported — interpret with care:


column
XEFYGUAN    0.0214
XEFYGUKN    0.1056
Name: share, dtype: float64

## 8. Decode categoricals

Labels from the published value sets, not hand-typed mappings.

In [8]:
CATEGORICALS = ['EFFYLEV']

unresolved = {}
for col in CATEGORICALS:
    if col in frame.columns:
        frame = iu.decode(frame, valuesets, col)
        unmatched = frame.loc[frame[col].notna() & frame[f"{col}_LABEL"].isna(), col].unique()
        if len(unmatched):
            unresolved[col] = sorted(unmatched.tolist())[:10]

# An unmatched code means a taxonomy change or a parsing fault. Either way the
# labels are wrong, so this stops the notebook rather than printing a warning.
assert not unresolved, f"codes absent from the published value set: {unresolved}"

label_cols = [c for c in frame.columns if c.endswith("_LABEL")]
frame[CATEGORICALS + label_cols].drop_duplicates().head(20) if label_cols else frame.head()

,EFFYLEV,EFFYLEV_LABEL
0,1,All students total
1,2,Undergraduate
2,<NA>,NaN
6,4,Graduate


## 9. Reshape to the declared grain

Target grain: `UNITID` x `EFFYALEV`. The grain is asserted, not assumed, because a duplicated key silently inflates every aggregate computed downstream.

In [9]:
curated = frame.copy()

# This component already arrives at its declared grain, so curation is a
# pass-through. Components with a long layout (GRTYPE, EFFYALEV, STAFFCAT,
# OMCHRT) filter or pivot here instead; see c10_f for a full worked reshape.

present_grain = [g for g in GRAIN if g in curated.columns]
duplicated = curated.duplicated(subset=present_grain, keep=False).sum()
print(f"grain {present_grain} -> {len(curated):,} rows, {duplicated} duplicated")
assert duplicated == 0, "Declared grain is not unique; resolve before continuing."

curated.head()

grain ['UNITID', 'EFFYALEV'] -> 116,437 rows, 0 duplicated


,UNITID,EFFYALEV,EFFYLEV,LSTUDY,EFYTOTLT,EFYTOTLM,EFYTOTLW,EFYGUKN,EFYGUAN,EFYAIANT,EFYASIAT,EFYBKAAT,EFYHISPT,EFYNHPIT,EFYWHITT,EFY2MORT,EFYUNKNT,EFYNRALT,EFFYLEV_LABEL
0,100654,1,1,999.0,6627.0,2644.0,3983.0,6622.0,NaN,16.0,18.0,5633.0,72.0,10.0,142.0,74.0,541.0,121.0,All students total
1,100654,2,2,1.0,5664.0,2329.0,3335.0,5662.0,NaN,13.0,13.0,5070.0,67.0,10.0,119.0,73.0,234.0,65.0,Undergraduate
2,100654,3,<NA>,1.0,5660.0,2329.0,3331.0,NaN,NaN,13.0,13.0,5067.0,67.0,10.0,118.0,73.0,234.0,65.0,NaN
3,100654,4,<NA>,1.0,1756.0,722.0,1034.0,NaN,NaN,3.0,2.0,1573.0,25.0,4.0,23.0,24.0,79.0,23.0,NaN
4,100654,5,<NA>,1.0,3904.0,1607.0,2297.0,NaN,NaN,10.0,11.0,3494.0,42.0,6.0,95.0,49.0,155.0,42.0,NaN


## 10. Validate

Rules are declarative so the output is a persistable report: which checks ran, which failed, on how many rows, and which institutions were implicated. That report is the artefact you cite when claiming this table is fit for analysis.

In [10]:
RULES = [
    iu.unique_key('UNITID', 'EFFYALEV'),
    iu.in_range('EFYTOTLT', 0, None),
    iu.sums_to('EFYTOTLT', ['EFYAIANT','EFYASIAT','EFYBKAAT','EFYHISPT','EFYNHPIT','EFYWHITT','EFY2MORT','EFYUNKNT','EFYNRALT'], severity='warn'),
]

report = iu.validate(curated, RULES, SLUG)
report.save(f"../reports/validation/{SLUG}.json")
display(report.to_frame()[["name", "status", "n_offending", "share", "note"]])

print("PASSED" if report.ok else "FAILED")
report.raise_if_failed()

,name,status,n_offending,share,note
0,"unique_key(UNITID,EFFYALEV)",pass,0,0.0,Declared grain must be unique
1,"in_range(EFYTOTLT,0,None)",pass,0,0.0,Value plausibility bound
2,sums_to(EFYTOTLT),pass,0,0.0,Parts must reconcile within 0


PASSED


Report(table='c03_e12', rows=116437, results=[{'name': 'unique_key(UNITID,EFFYALEV)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Declared grain must be unique', 'status': 'pass'}, {'name': 'in_range(EFYTOTLT,0,None)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Value plausibility bound', 'status': 'pass'}, {'name': 'sums_to(EFYTOTLT)', 'severity': 'warn', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Parts must reconcile within 0', 'status': 'pass'}], generated_utc='2026-09-24T17:19:15+00:00')

## 11. Write the curated table

The sidecar carries the reference period and grain with the data. This is the defence against assembling a panel by filename year when the underlying periods are offset differently per component.

In [11]:
path = iu.write_curated(
    curated,
    SLUG,
    root="../data/curated",
    reference_period=REFERENCE_PERIOD,
    grain=GRAIN,
    provenance=provenance,
    notes='Do not assume EFYTOTLM + EFYTOTLW == EFYTOTLT. Gender-inclusive reporting categories mean the men/women columns no longer partition the total. Any men/women share must use EFYGUKN as its denominator, or shares will quietly sum to under 100 percent at exactly the institutions using the newer categories.',
)

iu.write_provenance(provenance, f"../docs/provenance/{SLUG}.json")
print("wrote", path, f"({len(curated):,} rows x {curated.shape[1]} columns)")

wrote ../data/curated/c03_e12.parquet (116,437 rows x 19 columns)


## 12. Exercises

1. Re-run this notebook against the prior collection cycle by changing `TABLES`. The schema lock and the period assertion will both object; resolve each objection and record what changed between cycles.
2. Identify the three columns with the lowest reported-flag share, and argue whether each belongs in a predictive model at all.
3. Construct one derived cross-tabulation from this table, then apply `iu.suppress` and `iu.k_anonymity` to it. Report the smallest equivalence class before and after coarsening, and state the k you would require before publishing.
4. Do not assume EFYTOTLM + EFYTOTLW == EFYTOTLT. Write a validation rule that would catch this error if a colleague made it, and add it to `RULES` above.